In [0]:
# 1. Read from the Silver table as a stream or static batch view
silver_df = spark.read.table("realtime_fraud_detection.02_silver.cleaned_transactions")

# 2. Aggregate transactions by Time intervals (or hourly/window buckets) and Class status
from pyspark.sql.functions import col, count, sum, avg

gold_summary_df = (
    silver_df
    .groupBy("Class")
    .agg(
        count("*").alias("total_transactions"),
        sum("Amount").alias("total_amount"),
        avg("Amount").alias("average_amount")
    )
)

# 3. Write out to your Gold Delta table
# Batch Write Version
query = (
    gold_summary_df.write
    .format("delta")
    .mode("overwrite") # Overwrite or append for batch tables
    .saveAsTable("realtime_fraud_detection.03_gold.hourly_fraud_summary")
)

print("🥇 Gold Layer Analytics Summary Table Generated Successfully!")

In [0]:
display(spark.read.table("realtime_fraud_detection.03_gold.hourly_fraud_summary"))